In [ ]:
!pip -q install transformers faiss-cpu gradio

In [ ]:
import os
import numpy as np
import torch
import faiss
import gradio as gr
import matplotlib.pyplot as plt

from PIL import Image
from torchvision.datasets import CIFAR10
from transformers import CLIPProcessor, CLIPModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
os.makedirs("project_data/images", exist_ok=True)
print("Folder project_data/images is ready.")

In [ ]:
dataset = CIFAR10(root="project_data", train=False, download=True)
print("CIFAR-10 downloaded.")
print("Number of images in test split:", len(dataset))

In [ ]:
max_images = 500

for i in range(max_images):
    img, label = dataset[i]
    img.save(f"project_data/images/img_{i:04d}.png")

print(f"Saved {max_images} images into project_data/images")

In [ ]:
sample_path = "project_data/images/img_0000.png"
sample_img = Image.open(sample_path)

plt.imshow(sample_img)
plt.axis("off")
plt.title("Sample CIFAR Image")
plt.show()

In [ ]:
# load CLIP model (using pretrained, no training needed)

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

print("CLIP model loaded")

In [ ]:
# get all image paths from folder

image_folder = "project_data/images"

image_paths = []

for file in os.listdir(image_folder):
    if file.endswith(".png"):
        image_paths.append(os.path.join(image_folder, file))

# sort just to keep order consistent
image_paths = sorted(image_paths)

print("Total images:", len(image_paths))

In [ ]:
# function to convert one image into CLIP embedding

def get_image_embedding(img_path):
    img = Image.open(img_path).convert("RGB")
    
    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    
    # force tensor
    if isinstance(outputs, torch.Tensor):
        features = outputs
    else:
        features = outputs[0]
    
    # normalize
    features = features / torch.norm(features, dim=-1, keepdim=True)
    
    return features.cpu().numpy()

In [ ]:
# create embeddings for all images

all_embeddings = []

for i, path in enumerate(image_paths):
    emb = get_image_embedding(path)
    all_embeddings.append(emb[0])
    
    # print progress every 50 images (not too fancy)
    if (i + 1) % 50 == 0:
        print("Processed", i + 1, "images")

image_embeddings = np.array(all_embeddings).astype("float32")

print("finished creating embeddings")
print("Shape:", image_embeddings.shape)